# Advanced PySpark


## Collect_List

In [0]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.display()

In [0]:
from pyspark.sql.types  import *
from pyspark.sql.functions import *

In [0]:
df_book.groupBy('user').agg(collect_list('book')).display()

## pivot

In [0]:
df = (
    spark.read.format("csv")
    .option("inferSchema", True)
    .option("header", True)
    .load("/Volumes/dataxbootcamp/default/datax/BigMart Sales.csv")
)

df.display()

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

## When-Otherwise

In [0]:
df = spark.read.format('csv').option('header', True).load('/Volumes/dataxbootcamp/default/datax/BigMart Sales.csv')

from pyspark.sql.functions import col, when

df = df.withColumn(
    'veg_flag',
    when(
        col('Item_Type') == 'Meat',
        'Non-Veg'
    ).otherwise('Veg')
)

display(df)

In [0]:
df.withColumn('veg_exp_flag',when(((col('veg_flag')=='Veg') & (col('Item_MRP')<100)),'Veg_Inexpensive')\
                            .when((col('veg_flag')=='Veg') & (col('Item_MRP')>100),'Veg_Expensive')\
                            .otherwise('Non_Veg')).display() 

In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    'veg_exp_flag',
    when(
        (col('veg_flag') == 'Veg') & (col('Item_MRP').cast('double') < 100),
        'Veg_Inexpensive'
    ).when(
        (col('veg_flag') == 'Veg') & (col('Item_MRP').cast('double') > 100),
        'Veg_Expensive'
    ).otherwise('Non_Veg')
)

display(df)

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()
df2.display()


## Inner Join

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'inner').display()

## Left Join & Right Join

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'left').display()
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'right').display()

## Anti Join

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'anti').display()

# _Window Functions_

### ROw_number()

In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window

In [0]:
df.withColumn('reoCol', row_number().over(Window.orderBy('Item_Identifier'))).display()

## Rank() & Dense_Rank()

In [0]:
df.withColumn('rankCol', rank().over(Window.orderBy('Item_Identifier'))).display()
df.withColumn('denseRankCol', dense_rank().over(Window.orderBy('Item_Identifier'))).display()
df.withColumn('percentRankCol', percent_rank().over(Window.orderBy('Item_Identifier'))).display()

In [0]:
df.withColumn('dum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()